In [0]:
from pyspark.sql.functions import current_timestamp

In [0]:
# Preâmbulo para configurar o catálogo
catalog = dbutils.widgets.get("catalog")

bronze_schema = f"{catalog}.bronze"
landing_path = f"/Volumes/{catalog}/bronze/landing"

spark.sql(f"CREATE CATALOG IF NOT EXISTS {catalog}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {bronze_schema}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {bronze_schema}.landing")

print("Catálogo configurado com sucesso!")

In [0]:
# Função para carregar um DataFrame a partir do nome de um arquivo
def load_df(file_name):
    # Se der erro, a execução deve parar
    try:
        existing = {f.name for f in dbutils.fs.ls(landing_path)}
    except Exception:
        raise FileNotFoundError(f"A pasta {landing_path} não existe ou não está acessível.")
    if file_name not in existing:
        raise FileNotFoundError(
            f"{file_name} não foi encontrado em {landing_path}. "
            f"Arquivos disponíveis: {sorted(existing)}"
        )

    return spark.read.csv(f"{landing_path}/{file_name}", header=True, inferSchema=True)

In [0]:
# Função para salvar o DataFrame na camada bronze
def save_df(df, table_name):
    df.withColumn("ingestion_datetime", current_timestamp()) \
              .write.format("delta").mode("append") \
              .saveAsTable(f"{bronze_schema}.{table_name}")